# Step 6: Backtest

Would the mature-field method have worked in the past? Pretend it is June of an earlier year, use only the data known then, forecast the next 24 months, and compare with what actually happened.

Only the mature-group decline is tested: it is mechanical. The Sverdrup, Castberg and young-field cases rest on today's engineering judgement, so they cannot be backtested fairly.

Cutoffs start in June 2020. An earlier start would be distorted by Norway's government-ordered production cuts in 2020.

In [1]:
from getpass import getpass

import pandas as pd
from sqlalchemy import URL, create_engine, text

from forecast_tools import arps_forecast, fit_arps, mature_fields_as_of

# Connect straight to the project database; the password is asked for, never stored
db_url = URL.create(
    "mysql+pymysql",
    username="root",
    password=getpass("MySQL root password: "),
    host="localhost",
    port=3306,
    database="norway_forecast",
)
db = create_engine(db_url)

monthly = pd.read_sql(
    text("SELECT field_name, production_month, liquids_kbd FROM silver_field_production_rates"),
    db,
    parse_dates=["production_month"],
)
liquids_by_field = (
    monthly
    .pivot_table(index="production_month", columns="field_name", values="liquids_kbd", fill_value=0)
    .asfreq("MS", fill_value=0)
)
print(f"{liquids_by_field.shape[1]} fields, {liquids_by_field.index.min():%Y-%m} to {liquids_by_field.index.max():%Y-%m}")

136 fields, 1971-06 to 2026-07


In [2]:
CUTOFFS = ["2020-06-01", "2021-06-01", "2022-06-01", "2023-06-01", "2024-06-01"]
FIT_WINDOWS_MONTHS = [36, 60]                             # 3-year vs 5-year fit window
HORIZONS = {"months 1-12": range(1, 13), "months 13-24": range(13, 25)}

results = []
for cutoff in pd.to_datetime(CUTOFFS):
    # Group membership and fit use only data up to the cutoff...
    mature_fields = mature_fields_as_of(liquids_by_field, cutoff)
    mature_total = liquids_by_field[mature_fields].sum(axis=1)

    for window in FIT_WINDOWS_MONTHS:
        fit_start = cutoff - pd.DateOffset(months=window - 1)
        fit = fit_arps(mature_total[fit_start:cutoff], fit_start, b=0.01)

        # ...and are then compared with what those same fields actually produced
        for horizon, months_ahead in HORIZONS.items():
            months = pd.DatetimeIndex([cutoff + pd.DateOffset(months=m) for m in months_ahead])
            forecast = arps_forecast(fit, months).mean()
            actual = mature_total[months].mean()
            results.append({
                "cutoff": f"{cutoff:%Y-%m}",
                "fit_window": f"{window // 12} years",
                "horizon": horizon,
                "n_fields": len(mature_fields),
                "forecast_kbd": forecast,
                "actual_kbd": actual,
                "error_pct": (forecast / actual - 1) * 100,
            })

results = pd.DataFrame(results)
results.round(1)

,cutoff,fit_window,horizon,n_fields,forecast_kbd,actual_kbd,error_pct
0,2020-06,3 years,months 1-12,72,1226.5,1217.1,0.8
1,2020-06,3 years,months 13-24,72,1102.9,1134.6,-2.8
2,2020-06,5 years,months 1-12,72,1319.8,1217.1,8.4
3,2020-06,5 years,months 13-24,72,1229.1,1134.6,8.3
4,2021-06,3 years,months 1-12,72,1129.9,1103.5,2.4
5,2021-06,3 years,months 13-24,72,1010.9,1025.2,-1.4
6,2021-06,5 years,months 1-12,72,1199.3,1103.5,8.7
7,2021-06,5 years,months 13-24,72,1106.5,1025.2,7.9
8,2022-06,3 years,months 1-12,72,861.5,937.4,-8.1
9,2022-06,3 years,months 13-24,72,735.6,849.5,-13.4


In [3]:
# Error per cutoff, side by side (positive = forecast too high)
print(results.pivot_table(index="cutoff", columns=["fit_window", "horizon"], values="error_pct").round(1))

# Bias (average error) and accuracy (average size of the error, sign ignored)
results.groupby(["fit_window", "horizon"])["error_pct"].agg(
    bias_pct="mean",
    mean_abs_error_pct=lambda errors: errors.abs().mean(),
).round(1)

fit_window     3 years                  5 years             
horizon    months 1-12 months 13-24 months 1-12 months 13-24
cutoff                                                      
2020-06            0.8         -2.8         8.4          8.3
2021-06            2.4         -1.4         8.7          7.9
2022-06           -8.1        -13.4         1.2          0.3
2023-06            2.7          2.5         4.1          4.6
2024-06            1.4         -6.2         8.6          4.4


bias_pct  mean_abs_error_pct
fit_window horizon                                   
3 years    months 1-12       -0.2                 3.1
           months 13-24      -4.3                 5.3
5 years    months 1-12        6.2                 6.2
           months 13-24       5.1                 5.1